# P-Tuning semantic similarity 예제 - MRPC 데이터셋
## 작성자 : AISchool ([http://aischool.ai/](http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/) )
## Reference : https://huggingface.co/docs/peft/task_guides/ptuning-seq-classification


# 필요한 라이브러리 설치

In [ ]:
# datasets는 2.21.0으로 고정하고, accelerate를 추가했습니다.
!pip install -q -U peft transformers accelerate evaluate "datasets==2.21.0"

# 설정값 지정

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from peft import (
    get_peft_config,
    get_peft_model,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
    PeftType,
    PromptEncoderConfig,
)
from datasets import load_dataset
import evaluate
import torch

model_name_or_path = "roberta-large"
task = "mrpc" #MRPC 데이터 셋 사용
num_epochs = 5
lr = 1e-3 #Learning Rate 0.001
batch_size = 32

# MRPC 데이터셋 불러오기
## (GLUE 벤치마크 데이터셋의 부분집합 데이터셋으로 **두개의 문장이 의미론적으로 동일한지 다른지를 측정**하는 데이터셋입니다.)

In [ ]:
dataset = load_dataset("glue", task)
dataset["train"][0]

# F1 scroe 측정을 위한 metric을 정의합니다.

In [ ]:
metric = evaluate.load("glue", task)

In [ ]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

# Tokenizer 불러오기 & 데이터셋 전처리

In [ ]:
if any(k in model_name_or_path for k in ("gpt", "opt", "bloom")):
    padding_side = "left"
else:
    padding_side = "right"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, padding_side=padding_side)
if getattr(tokenizer, "pad_token_id") is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

#토크나이징
def tokenize_function(examples):
    # max_length=None => use the model max length (it's actually the default) #Max_length를 설정하지 않으면 모델 자체에 있는 max_length에 맞춰진다
    outputs = tokenizer(examples["sentence1"], examples["sentence2"], truncation=True, max_length=None)
    return outputs

In [ ]:
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["idx", "sentence1", "sentence2"],
)

tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

# PEFT 모델 설정

In [ ]:
peft_config = PromptEncoderConfig(task_type="SEQ_CLS", num_virtual_tokens=20, encoder_hidden_size=128)

model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, return_dict=True)

# P-Tuning 기법으로 인해 전체 모델의 0.67%의 파라미터만 Fine-Tuning에 사용

In [ ]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Training 시작

In [ ]:
training_args = TrainingArguments(
    output_dir="roberta-large-peft-p-tuning",
    learning_rate=lr, #Learning Rate 설정
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size, #Batch_size 설정
    num_train_epochs=num_epochs, #Train Epoch 설정
    weight_decay=0.01, #가중치 설
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
#Validation Loss = 모델이 얼마나 틀렸는지에 대한 오답률 (해당 값은 낮을 수록 좋다)
#Accuracy = 정답률 (해당 값은 높을 수록 좋)
#F1 = 정확도에 대한 함정 데이터들을 보완환 실제 점수 (0.8 이상이면 꽤나 준수한 성)

# 학습이 끝난 모델 불러오기

In [ ]:
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

peft_model_id = "./roberta-large-peft-p-tuning/checkpoint-575"
config = PeftConfig.from_pretrained(peft_model_id)
inference_model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path)

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = PeftModel.from_pretrained(inference_model, peft_model_id)

# 학습이 끝난 모델을 sample text에 대한 Inference

In [ ]:
classes = ["not equivalent", "equivalent"]

sentence1 = "Coast redwood trees are the tallest trees on the planet and can grow over 300 feet tall."
sentence2 = "The coast redwood trees, which can attain a height of over 300 feet, are the tallest trees on earth."

inputs = tokenizer(sentence1, sentence2, truncation=True, padding="longest", return_tensors="pt")

In [ ]:
with torch.no_grad():
    outputs = model(**inputs).logits
    print(outputs)

paraphrased_text = torch.softmax(outputs, dim=1).tolist()[0]
for i in range(len(classes)):
    print(f"{classes[i]}: {int(round(paraphrased_text[i] * 100))}%")